# Análisis de Sueldos vs. Costo de Vida en Chile (2022–2025)
**Autor:** Claudio Galiano · **Metodología:** CRISP-DM · **Herramientas:** Python (Google Colab) y Power BI

**Objetivo:** analizar la relación entre los ingresos laborales y el costo de vida en Chile, identificando diferencias por región, género, educación y sector económico, y sus posibles causas.

Estructura según la metodología **CRISP-DM** (Cross-Industry Standard Process for Data Mining):

| Fase | Contenido | Estado |
|---|---|---|
| 1 | Comprensión del negocio | Listo |
| 2 | Comprensión de los datos | En curso (configuración y carga listas, falta exploración) |
| 3 | Preparación de los datos | Pendiente |
| 4 | Modelado | Pendiente |
| 5 | Evaluación | Pendiente |
| 6 | Despliegue y conclusiones | Pendiente |

## 1. Comprensión del negocio

### 1.1 Contexto
En los últimos años el costo de la vida en Chile ha sido uno de los temas que más preocupa a las personas. Entre 2022 y 2023 la inflación llegó a niveles que no se veían en décadas, y el precio de la vivienda siguió subiendo. Al mismo tiempo, los sueldos también aumentaron, pero no de la misma forma en todas las regiones, sectores ni grupos de trabajadores. La pregunta de fondo es simple: ¿el sueldo alcanza igual que antes, y alcanza igual para todos?

### 1.2 Problema
No basta con mirar el sueldo promedio del país. Un mismo ingreso no rinde lo mismo en Santiago que en La Araucanía, ni para un hombre que para una mujer con la misma formación. Este trabajo busca medir esas diferencias con datos oficiales y entender qué las explica.

### 1.3 Objetivo general
Analizar la relación entre los ingresos laborales y el costo de vida en Chile entre 2022 y 2025, identificando diferencias territoriales, de género, educacionales y sectoriales, y sus posibles causas, mediante técnicas de minería de datos y un modelo OLAP.

### 1.4 Objetivos específicos
1. Integrar datos oficiales de ingresos (ESI), precios (IPC) y vivienda (CASEN) en un modelo dimensional.
2. Comparar la evolución del ingreso real con la inflación del período.
3. Identificar brechas de ingreso por región, sexo, nivel educacional y sector económico.
4. Agrupar regiones con perfiles similares y determinar qué factores explican mejor las diferencias de ingreso.
5. Presentar los resultados en un dashboard interactivo en Power BI.

### 1.5 Preguntas de valor
1. ¿Creció más el sueldo o el costo de vida entre 2022 y 2025?
2. ¿En qué regiones rinde más el sueldo si se considera lo que cuesta arrendar?
3. ¿Cuál es la brecha de ingresos entre hombres y mujeres, y cambia según la educación, la edad o el sector?
4. ¿Cuánto aumenta el ingreso con cada nivel educacional?
5. ¿Qué tipo de gasto explica más el alza del costo de vida?
6. ¿Qué factor pesa más en el ingreso: la región, el sector, la educación, el sexo o la edad?

### 1.6 Hipótesis iniciales
| Hipótesis | Cómo se pondrá a prueba |
|---|---|
| Las regiones mineras tienen ingresos más altos por el peso de la minería | Ingreso por región y por sector |
| La inflación de 2022 redujo el poder de compra | Ingreso nominal frente a ingreso real |
| La vivienda subió más que los sueldos | IPC de vivienda frente a la variación del ingreso |
| Parte de la brecha de género se explica por el tipo de trabajo | Brecha dentro de un mismo sector y nivel educacional |
| La informalidad baja los ingresos | Ingreso de trabajadores formales e informales |

### 1.7 Criterios de éxito
- Las cifras calculadas coinciden con las publicadas por el INE.
- Cada pregunta de valor queda respondida con datos y un gráfico.
- El modelo dimensional permite filtrar por año, región, sexo, educación y sector.

### 1.8 Recursos
- **Datos:** INE (ESI 2022–2025 e IPC), Banco Central (series del IPC) y Ministerio de Desarrollo Social (CASEN 2024).
- **Herramientas:** Python en Google Colab, Power BI, Jira para la planificación y GitHub para el control de versiones.

> **Observación.** Una limitación conocida desde el inicio es que Chile no publica un IPC por región. Para comparar el costo de vida entre regiones se usará el arriendo mediano de la CASEN como aproximación, y así se indicará en las conclusiones.

## 2. Comprensión de los datos

### 2.1 Configuración del entorno
Antes de trabajar hay que preparar el "escritorio": conectar Google Drive (donde están los datos), cargar las herramientas de Python y decirle al notebook dónde están las carpetas.

#### 2.1.1 Conectar Google Drive
**Qué hace:** le da permiso a Colab para leer los archivos de tu Drive. Al ejecutarla aparece una ventana: elige tu cuenta y acepta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### 2.1.2 Cargar librerías
**Qué hace:** las librerías son "cajas de herramientas" que otras personas ya programaron:
- **os** → maneja carpetas y rutas de archivos.
- **zipfile** → abre archivos comprimidos (.zip), como el de la CASEN.
- **numpy** → cálculos numéricos.
- **pandas** → trabaja con tablas de datos (como un Excel dentro de Python). Es la principal.
- **matplotlib / seaborn** → hacen gráficos.
- **pyreadstat** → permite leer archivos SPSS (.sav), el formato de la CASEN. No viene instalada en Colab, por eso se instala con `!pip install`.

In [ ]:
!pip install -q pyreadstat

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Mostrar todas las columnas y los números con separador de miles
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Librerías cargadas')

#### 2.1.3 Rutas del proyecto
**Qué hace:** guarda en variables la ubicación de las carpetas, para no escribirla completa cada vez. Luego revisa que los archivos estén donde deben.

> Si moviste la carpeta del proyecto en Drive, solo cambia la línea `RUTA`.

In [ ]:
RUTA = '/content/drive/MyDrive/Proyecto_Ingresos_Chile/'
CRUDOS = os.path.join(RUTA, '01_datos_crudos')

for carpeta in ['esi', 'ipc', 'casen']:
    archivos = os.listdir(os.path.join(CRUDOS, carpeta))
    print(f'{carpeta}: {archivos}')

### 2.2 Carga de datos
Aquí se **leen** los archivos originales y se guardan en tablas de pandas (llamadas *DataFrames*). Todavía **no se limpia ni se cambia nada**: solo se cargan para conocerlos en la exploración (2.3).

| Tabla | Fuente | Qué contiene |
|---|---|---|
| `esi` | INE · Encuesta Suplementaria de Ingresos 2022–2025 | Una fila por persona: región, sexo, edad, educación, trabajo e **ingreso** |
| `ipc_general` | INE vía Banco Central | IPC mensual general (mide cuánto suben los precios) |
| `ipc_divisiones` | INE vía Banco Central | IPC mensual de 13 tipos de gasto (alimentos, vivienda, transporte…) |
| `ipc_ponderadores` | INE vía Banco Central | Cuánto pesa cada tipo de gasto en el presupuesto de los hogares |
| `casen` | Ministerio de Desarrollo Social · CASEN 2024 | Una fila por persona; incluye cuánto paga su hogar de **arriendo** |

#### 2.2.1 ESI 2022–2025 (ingresos)
**Qué hace:**
1. Recorre los 4 años con un ciclo `for`.
2. Lee cada archivo CSV con `pd.read_csv`.
3. Le agrega una columna `anio` para saber de qué año es cada fila.
4. Une los 4 años en una sola tabla con `pd.concat`.

Tarda unos segundos porque cada archivo pesa ~80 MB.

In [ ]:
anios = [2022, 2023, 2024, 2025]
lista_esi = []

for anio in anios:
    archivo = os.path.join(CRUDOS, 'esi', f'esi_{anio}.csv')
    df_anio = pd.read_csv(archivo, low_memory=False)
    df_anio['anio'] = anio
    lista_esi.append(df_anio)
    print(f'ESI {anio}: {df_anio.shape[0]:,} filas · {df_anio.shape[1]} columnas')

esi = pd.concat(lista_esi, ignore_index=True)
print(f'\nESI total: {esi.shape[0]:,} filas · {esi.shape[1]} columnas')

**Primer vistazo:** `head()` muestra las primeras 5 filas. Verás que casi todo son **números**: muchos son **códigos** (por ejemplo, en `sexo`, 1 = Hombre y 2 = Mujer). Su significado está en los libros de códigos (`02_documentacion_fuentes`).

In [ ]:
esi.head()

> **Observación.** Los cuatro años suman 369.731 registros. La muestra de 2024 es la más chica (85.172 personas), así que conviene tenerlo presente al comparar ese año. El archivo de 2025 trae 290 columnas y los anteriores 284, porque el INE incorporó preguntas nuevas; al unirlos quedan 292 columnas y las nuevas aparecen vacías en los años que no las tenían. Casi todas las variables vienen como códigos numéricos, por lo que el libro de códigos es indispensable para interpretarlas.

#### 2.2.2 IPC (precios)
**Qué hace:** lee 3 archivos Excel descargados del Banco Central. Todos tienen la hoja `Cuadro` y 2 filas de título arriba, por eso se usa `skiprows=2` (salta esas filas).

- **IPC general:** una fila por mes, de enero 2022 a diciembre 2025.
- **IPC por división:** una fila por mes y una columna por tipo de gasto (desde 2023).
- **Ponderadores:** el peso (%) de cada tipo de gasto en la canasta.

In [ ]:
ipc_general = pd.read_excel(os.path.join(CRUDOS, 'ipc', 'ipc_general_empalmado_2022_2025.xlsx'),
                            sheet_name='Cuadro', skiprows=2)
ipc_divisiones = pd.read_excel(os.path.join(CRUDOS, 'ipc', 'ipc_divisiones_2023_2025.xlsx'),
                               sheet_name='Cuadro', skiprows=2)
ipc_ponderadores = pd.read_excel(os.path.join(CRUDOS, 'ipc', 'ipc_ponderadores_divisiones_2023.xlsx'),
                                 sheet_name='Cuadro', skiprows=2)

print(f'IPC general:     {ipc_general.shape[0]} filas · {ipc_general.shape[1]} columnas')
print(f'IPC divisiones:  {ipc_divisiones.shape[0]} filas · {ipc_divisiones.shape[1]} columnas')
print(f'Ponderadores:    {ipc_ponderadores.shape[0]} fila · {ipc_ponderadores.shape[1]} columnas')
ipc_general.head()

> **Observación.** El IPC general cubre los 48 meses entre enero de 2022 y diciembre de 2025, sin vacíos. Las divisiones parten recién en 2023 porque ese año el INE cambió la canasta y pasó de 12 a 13 divisiones; mezclar ambas bases no sería correcto. Según los ponderadores, alimentos (22,2 %), vivienda (16,8 %) y transporte (13,5 %) explican más de la mitad del gasto de los hogares, así que cualquier alza en esos tres rubros se nota mucho más en el bolsillo.

#### 2.2.3 CASEN 2024 (arriendo)
**Qué hace:**
1. Descomprime el archivo `.zip` (adentro viene `casen_2024.sav`, formato SPSS).
2. Lo lee con `pd.read_spss`, pero **solo las 7 columnas que necesitamos**: la base completa tiene cientos de columnas y podría colapsar Colab.

| Columna | Significado |
|---|---|
| `folio` | Número que identifica a cada hogar |
| `region` | Región (1 a 16, mismos códigos que la ESI) |
| `area` | Urbano o rural |
| `ten_viv` | Tenencia de la vivienda (2 = arrendada) |
| `v18` | Cuánto paga el hogar de arriendo (-88 = no sabe) |
| `v19` | Cuánto se paga de arriendo en el sector por viviendas parecidas |
| `expr` | Factor de expansión: a cuántas personas reales representa cada fila |

In [ ]:
zip_casen = os.path.join(CRUDOS, 'casen', 'casen_2024spss.zip')
carpeta_temporal = '/content/casen'
os.makedirs(carpeta_temporal, exist_ok=True)

with zipfile.ZipFile(zip_casen) as z:
    archivo_sav = [n for n in z.namelist() if n.lower().endswith('.sav')][0]
    z.extract(archivo_sav, carpeta_temporal)

columnas_casen = ['folio', 'region', 'area', 'ten_viv', 'v18', 'v19', 'expr']
casen = pd.read_spss(os.path.join(carpeta_temporal, archivo_sav),
                     usecols=columnas_casen, convert_categoricals=False)

print(f'CASEN 2024: {casen.shape[0]:,} personas · {casen["folio"].nunique():,} hogares')
casen.head()

> **Observación.** La CASEN viene con una fila por persona, pero el arriendo es un dato del hogar: si no se corrige, un hogar de cinco personas pesaría cinco veces. Por eso, antes de calcular, habrá que dejar una sola fila por cada folio. El monto de arriendo (v18) solo lo responden los hogares que arriendan, y el código -88 corresponde a "no sabe", que debe excluirse.

#### 2.2.4 Resumen de lo cargado
**Qué hace:** junta en una tabla cuántas filas y columnas tiene cada fuente, para confirmar que todo se cargó.

In [ ]:
resumen = pd.DataFrame({
    'tabla': ['esi', 'ipc_general', 'ipc_divisiones', 'ipc_ponderadores', 'casen'],
    'filas': [len(esi), len(ipc_general), len(ipc_divisiones), len(ipc_ponderadores), len(casen)],
    'columnas': [esi.shape[1], ipc_general.shape[1], ipc_divisiones.shape[1],
                 ipc_ponderadores.shape[1], casen.shape[1]],
})
resumen

> **Observación general.** Las tres fuentes no tienen el mismo nivel de detalle: la ESI describe personas en un año, el IPC describe meses y la CASEN describe hogares. No se pueden unir fila por fila; la forma correcta de cruzarlas es a través de lo que tienen en común, la región y el tiempo. Esa es la base del modelo dimensional que se construirá más adelante.

### 2.3 Exploración de los datos
*Pendiente:* estructura y tipos de variables, calidad (faltantes y códigos especiales), distribución del ingreso, valores extremos y validación contra las cifras del INE.

## 3. Preparación de los datos
*Pendiente:* filtrar ocupados, recodificar variables con los libros de códigos, calcular el ingreso real con el IPC y resumir el arriendo por región con la CASEN.

## 4. Modelado
*Pendiente:* modelo dimensional de constelación (cubo OLAP), agrupamiento de regiones (K-Means) y regresión de factores del ingreso.

## 5. Evaluación
*Pendiente:* validación de resultados, métricas de los modelos y contraste de las hipótesis.

## 6. Despliegue y conclusiones
*Pendiente:* exportación del modelo para Power BI, dashboard y respuesta a las preguntas de valor.